# Crime Data Exploratory Data Analysis

This notebook provides an exploratory analysis of crime data stored in the database.

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
# Cell 2: Connect to database and load data
# Get database URL from environment or use default
DATABASE_URL = os.environ.get('DATABASE_URL', 'sqlite:///crime_data.db')
engine = create_engine(DATABASE_URL)

# Load crimes table into DataFrame
df = pd.read_sql_table('crimes', con=engine)

print(f"Loaded {len(df)} crime records from database")
print(f"Database: {DATABASE_URL}")
df.head()

In [ ]:
# Cell 3: Basic statistics
print("=== DataFrame Info ===")
df.info()
print("\n=== Statistical Summary ===")
df.describe()
print("\n=== Null Value Counts ===")
print(df.isnull().sum())

In [ ]:
# Cell 4: Crime by Type Bar Chart
crime_counts = df['type'].value_counts().head(15)

plt.figure(figsize=(12, 6))
sns.barplot(x=crime_counts.values, y=crime_counts.index, palette='viridis')
plt.title('Top 15 Crime Types', fontsize=16, fontweight='bold')
plt.xlabel('Number of Incidents', fontsize=12)
plt.ylabel('Crime Type', fontsize=12)
plt.tight_layout()
plt.show()

print("\nTop 5 Crime Types:")
print(crime_counts.head())

In [ ]:
# Cell 5: Crime by Month Line Chart
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.to_period('M')

monthly_crimes = df.groupby('month').size()

plt.figure(figsize=(14, 6))
plt.plot(monthly_crimes.index.astype(str), monthly_crimes.values, marker='o', linewidth=2)
plt.title('Monthly Crime Trends', fontsize=16, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Number of Crimes', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Crime by District Bar Chart
district_counts = df['district'].value_counts().head(15)

plt.figure(figsize=(12, 6))
sns.barplot(x=district_counts.values, y=district_counts.index, palette='magma')
plt.title('Top 15 Districts by Crime Count', fontsize=16, fontweight='bold')
plt.xlabel('Number of Incidents', fontsize=12)
plt.ylabel('District', fontsize=12)
plt.tight_layout()
plt.show()

print(f"\nTotal unique districts: {df['district'].nunique()}")

In [ ]:
# Cell 7: Correlation Heatmap (Numeric Columns)
# Select only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# Drop columns with all unique values (like IDs)
numeric_df = numeric_df[['latitude', 'longitude'] + [c for c in numeric_df.columns if c not in ['latitude', 'longitude', 'id']]]

plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap of Numeric Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Scatter Plot of Lat/Lng Points
# Filter out rows with missing coordinates
geo_df = df.dropna(subset=['latitude', 'longitude'])

plt.figure(figsize=(12, 10))
plt.scatter(geo_df['longitude'], geo_df['latitude'], alpha=0.3, s=5, c='red')
plt.title('Crime Locations (Lat/Lng)', fontsize=16, fontweight='bold')
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTotal records with coordinates: {len(geo_df)}")

In [ ]:
# Cell 9: Distribution Plots
# Crime type distribution (pie chart for top 8)
top_types = df['type'].value_counts().head(8)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart
axes[0].pie(top_types.values, labels=top_types.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Crime Type Distribution', fontsize=14, fontweight='bold')

# Hour distribution
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce').dt.hour
hour_counts = df['time'].value_counts().sort_index()
axes[1].bar(hour_counts.index, hour_counts.values, color='steelblue')
axes[1].set_title('Crimes by Hour of Day', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Hour (0-23)')
axes[1].set_ylabel('Number of Crimes')
axes[1].set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.show()

# Save summary statistics
print("\n=== Summary Statistics ===")
print(f"Total crime records: {len(df)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique crime types: {df['type'].nunique()}")
print(f"Unique districts: {df['district'].nunique()}")